[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Reinforcement_Learning.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Reinforcement Learning

Learning from *consequences* instead of labels — the paradigm the [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) name-dropped as RLHF and this course delivers. Five sessions: bandits, MDPs & Bellman, temporal-difference learning, policy gradients, and the road to RLHF — every algorithm verified against an exactly-solvable environment.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb) (expectations, LLN).
- [Training Dynamics](./Training_Dynamics.ipynb) for Session 4.
- Kinship worth knowing: TD learning's *new = old + α·(surprise)* is the [adaptive-filter heartbeat](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) yet again.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 5 — *Bandits: Exploration vs Exploitation* (~35 min)
**Goal:** the RL problem with no states: regret, ε-greedy, and UCB's optimism.
**Feeds into:** Session 2 (MDPs).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Bandits — Exploration vs Exploitation</b></summary>

**Timing (~35 min).** 8 min the paradigm shift · 10 min regret as the metric · 12 min the three policies · 5 min the regret-rate result.

**Open by naming what changed from every previous workshop.** Supervised learning hands you the right answer for each input. Here **nobody tells you what you should have done** — you only see the consequence of what you did, and the counterfactual is gone forever. That single difference generates everything hard about RL, and bandits isolate it with no states, no time, no credit assignment.

**Define regret before running anything, because it is the metric the whole session is about.** Regret is what you lost by not always pulling the best arm. **Reward alone is uninterpretable** — a good policy on a bad problem scores worse than a bad policy on a good one — but regret against the per-instance optimum is comparable across problems. Note the code averages over **800 random bandit instances**, which is what makes the curves smooth enough to read.

**Have the room predict all three curves before running.** Greedy: a flat line after an early rise, because it locks onto whichever arm looked good first and never checks. ε-greedy: a **straight line**, because it wastes $\varepsilon$ of every pull forever, including after it knows the answer. UCB: a curve that **flattens without stopping**. Getting three different shapes from three one-line policies is the payoff.

**The rates are the real content, so state them.** Greedy has **linear** regret (constant probability of being locked on the wrong arm). ε-greedy has **linear** regret at rate $\varepsilon \times$ average gap. UCB has **logarithmic** regret — provably optimal up to constants. **Linear versus logarithmic is not a tuning difference; it is a different asymptotic class**, and no ε schedule fixed in advance beats it.

**Explain the UCB bonus term as a confidence interval, because it demystifies the formula.** $\hat\mu_a + c\sqrt{\ln t / n_a}$ is "my estimate, plus how wrong it could plausibly be". The $1/\sqrt{n_a}$ is the standard error from [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb); the $\ln t$ makes the interval a *simultaneous* bound over time. **Optimism in the face of uncertainty**: an arm you have barely tried gets a large bonus and therefore gets tried, and the bonus shrinks automatically as evidence accumulates. Exploration is *scheduled by the data* rather than by a hyperparameter.

**Point at the update rule and name it, because it is about to reappear three more times.** `Q[a] += (reward - Q[a]) / N[a]` is an incremental mean: **new = old + α × (surprise)**. It is the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb), it will be the TD update in Session 3, and it is the same shape as every stochastic-approximation algorithm in the curriculum. **One line, four workshops.**

**Close with the honest caveat about UCB's constant.** $c = 2.0$ is a choice, and a badly chosen $c$ makes UCB explore far too much or too little. The *rate* is what is provably optimal; the constant still has to be set, and on a short horizon a well-tuned ε-greedy can beat a poorly-tuned UCB. **Asymptotic optimality is not the same as being better at $T = 1000$**, and the plot happens to show a regime where it is.
</details>

## 2. The Ten-Armed Testbed

💡 **Intuition.** Ten slot machines, unknown payouts, 1000 pulls: every pull spent *learning* is a pull not spent *earning*. That tension — exploration vs exploitation — is RL's signature dilemma, isolated from everything else. **ε-greedy** explores blindly and forever; **UCB** explores *strategically*: pull the arm whose plausible upside $\hat\mu_a + c\sqrt{\ln t / n_a}$ is highest — 'optimism in the face of uncertainty', with the bonus shrinking exactly like a [confidence interval](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [2]:
def bandit_run(policy, T=1000, K=10, runs=800):
    regret = np.zeros(T)
    for r in range(runs):
        mu = rng.standard_normal(K)
        best = mu.max()
        Q, N = np.zeros(K), np.zeros(K)
        for t in range(T):
            a = policy(Q, N, t)
            reward = mu[a] + rng.standard_normal()
            N[a] += 1; Q[a] += (reward - Q[a]) / N[a]          # incremental mean (the heartbeat)
            regret[t] += best - mu[a]
    return np.cumsum(regret / runs)

eps_greedy = lambda eps: (lambda Q, N, t: rng.integers(len(Q)) if rng.random() < eps else int(np.argmax(Q)))
def ucb(Q, N, t):
    if (N == 0).any(): return int(np.argmax(N == 0))
    return int(np.argmax(Q + 2.0 * np.sqrt(np.log(t + 1) / N)))

plt.figure(figsize=(8, 3))
for name, pol in [("greedy (ε=0)", eps_greedy(0)), ("ε=0.1", eps_greedy(0.1)), ("UCB", ucb)]:
    r = bandit_run(pol)
    plt.plot(r, label=f"{name}: total regret {r[-1]:.0f}")
plt.legend(); plt.xlabel("pull"); plt.ylabel("cumulative regret")
plt.title("greedy plateaus on a wrong arm; ε keeps paying tax; UCB's tax shrinks")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2966289/3262936799.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three one-line policies, 1000 pulls, averaged over 800 random bandits — and three **qualitatively different curve shapes**, which is more informative than the totals in the legend.

- **Greedy** rises fast, then goes **flat** — and flat is bad news here, because a flat cumulative-regret curve at a nonzero slope means it has stopped improving. It locked onto whichever arm looked good after one pull and never checked another. With 10 arms it lands on the true best about 1 time in 10 and pays the gap forever the rest of the time.
- **ε-greedy** is a **straight line**. It keeps learning, but it also keeps paying: 10% of every pull is spent on a uniformly random arm, *including long after it knows the answer*. Constant tax, linear regret.
- **UCB** bends over and keeps bending. Its exploration cost **shrinks with evidence**.

**Those shapes are asymptotic classes, not tuning differences.** Greedy and ε-greedy both have **linear** regret; UCB has **logarithmic** regret and is provably optimal up to constants. **No fixed ε achieves a logarithmic rate** — you would need a decaying schedule, and even then tuning it is exactly the problem UCB solves automatically.

**The mechanism is in the bonus term, and it is a confidence interval wearing a different hat.** $\hat\mu_a + c\sqrt{\ln t / n_a}$ says "my estimate, plus how wrong I could plausibly be". The $1/\sqrt{n_a}$ is the standard error of a sample mean; the $\ln t$ widens it enough to hold simultaneously across all times. **Optimism in the face of uncertainty**: an arm with few pulls has a large bonus, so it gets pulled; the bonus then shrinks on its own. **Exploration is scheduled by the data, not by a hyperparameter** — which is precisely what ε-greedy cannot do.

**Note the line doing the learning, because it will reappear three more times in this workshop.** `Q[a] += (reward - Q[a]) / N[a]` is an incremental mean, and it has the shape **new = old + α × (surprise)**. That is the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb), it becomes the TD update in Session 3, and it is the generic form of stochastic approximation. **One line, four workshops.**

**Two honest caveats before generalising the ranking.** First, $c = 2.0$ is a *choice*: UCB's optimality is about the **rate**, and the constant still needs setting — a poorly tuned UCB can lose to a well-tuned ε-greedy over 1000 pulls. Second, the arms here are stationary Gaussians with known noise scale. **On non-stationary problems UCB's shrinking bonus is a liability**, because it stops exploring an arm whose payoff has since changed, and that is a real failure mode in recommender systems.

**Finally, keep the framing in view: this is RL with everything removed except one dilemma.** No states, no transitions, no delayed reward, no credit assignment. **Every pull spent learning is a pull not spent earning**, and that tension survives into every later session — it is just harder to see once states and time are added back.

---
### 🕐 Session 2 of 5 — *MDPs & the Bellman Equation* (~40 min)
**Goal:** add states and time; solve a gridworld EXACTLY by dynamic programming.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (TD learning).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: MDPs & the Bellman Equation</b></summary>

**Timing (~40 min).** 10 min what states add · 12 min the Bellman equation as a fixed point · 10 min value iteration as a contraction · 8 min reading the policy.

**Name what changes from bandits in one sentence: actions now have consequences that persist.** A bandit pull affects only that pull's reward. A gridworld step changes *where you are*, and therefore every reward available afterwards. **That is the credit-assignment problem**, and the Bellman equation is its solution.

**Present the Bellman equation as self-consistency rather than as a definition to memorise.** Today's value = today's reward + $\gamma \times$ tomorrow's value. That is not a computation, it is a **constraint** the true value function must satisfy at every state simultaneously — 16 equations in 16 unknowns. Value iteration simply applies the constraint repeatedly until nothing moves.

**Make the contraction argument explicit, because it is the connection to the analysis track.** The Bellman operator is a $\gamma$-contraction in the sup norm: applying it to two value functions brings them at least $\gamma$ times closer. By the same completeness argument as [Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb), a contraction on a complete space has a **unique fixed point**, and iteration reaches it. **The discount factor is not a modelling preference; it is what makes the problem well-posed.** At $\gamma = 1$ with no terminal state the values can diverge.

**Then flag the number that does not match the naive bound, because a sharp student will notice.** 32 sweeps to reach $10^{-12}$ implies a per-sweep factor near 0.42, not $\gamma = 0.95$. Ask why. **The answer is the absorbing terminal states**: once an episode ends, no future value flows back, so the effective contraction is $\gamma \times P(\text{not terminating})$, which is much stronger than $\gamma$ alone. **The $\gamma$ bound is worst-case; termination beats it.**

**Point at the slip probability as the thing that makes this an MDP rather than a puzzle.** With `slip = 0.1` the intended move fails 10% of the time. That is why the transition model is a *distribution*, why values are *expected* returns, and why the optimal policy is not simply the shortest path. **Remove the slip and dynamic programming degenerates into graph search.**

**Read the printed policy as a result to be checked, not admired.** Every arrow should be defensible: states near the pit should route around it, and the cost of the detour should be visible in the value function. Ask the room to predict the arrow at $(1,1)$ — directly above the pit — before showing it. **A policy you can argue with is a policy you understand.**

**Close by naming why this session is structurally important to the whole workshop.** Value iteration needs the **transition model** — every probability, every reward, known in advance. No real agent has that. But because we *do* have it here, we get an **exact oracle**, and every learning algorithm in Sessions 3 and 4 is graded against it. **Almost no RL tutorial can check its own answers**, and that is exactly why this notebook uses a $4\times4$ grid rather than something impressive.
</details>

## 3. Markov Decision Processes

💡 **Intuition.** Now actions have *consequences that persist*: an MDP is states, actions, transition probabilities, rewards, and a discount $\gamma$. The value $V^\pi(s)$ is expected discounted return — and Bellman's equation says value is **recursively self-consistent**: today's value = today's reward + γ·tomorrow's value. The optimal version ($V^* = \max_a [r + \gamma E V^*]$) is a fixed-point equation, and **value iteration** just applies it until it stops moving — a contraction ([Sequences & Series](../Intro_Math/Analysis/Numerical_Sequences_and_Series.ipynb): Cauchy convergence with rate γ!). This gives us an *exact oracle* to test every learning algorithm against.

In [3]:
# 4x4 gridworld: start anywhere, goal at (3,3) reward +1, pit at (1,2) reward −1, step −0.02
SIZE, GOAL, PIT = 4, 15, 6
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
gamma, slip = 0.95, 0.1                                  # 10% chance the move slips sideways

def step_model(s, a):
    """returns list of (prob, s', r, done)"""
    if s in (GOAL, PIT): return [(1.0, s, 0.0, True)]
    out = []
    for prob, ai in [(1-slip, a), (slip/2, (a+2)%4), (slip/2, (a+3)%4 if a%2 else (a+1)%4)]:
        r0, c0 = divmod(s, SIZE)
        dr, dc = ACTIONS[ai]
        r1, c1 = min(max(r0+dr,0),SIZE-1), min(max(c0+dc,0),SIZE-1)
        s1 = r1*SIZE + c1
        rew = 1.0 if s1 == GOAL else (-1.0 if s1 == PIT else -0.02)
        out.append((prob, s1, rew, s1 in (GOAL, PIT)))
    return out

# value iteration = the exact solution (our ORACLE for everything later)
V = np.zeros(16)
for it in range(500):
    V_new = np.array([max(sum(p*(r + gamma*V[s1]*(not d)) for p, s1, r, d in step_model(s, a))
                          for a in range(4)) for s in range(16)])
    if np.abs(V_new - V).max() < 1e-12: break
    V = V_new
Q_star = np.array([[sum(p*(r + gamma*V[s1]*(not d)) for p, s1, r, d in step_model(s, a))
                    for a in range(4)] for s in range(16)])
pi_star = Q_star.argmax(1)
print(f"value iteration converged in {it} sweeps (contraction at rate γ={gamma})")
arrows = np.array(["↑","↓","←","→"])[pi_star].reshape(4,4)
arrows[3,3] = "G"; arrows[1,2] = "P"
print("optimal policy:\n", arrows)

value iteration converged in 32 sweeps (contraction at rate γ=0.95)
optimal policy:
 [['↓' '↓' '→' '↓']
 ['↓' '↓' 'P' '↓']
 ['→' '→' '→' '↓']
 ['→' '→' '→' 'G']]


**What just happened.** Value iteration converged in **32 sweeps** and produced a policy whose arrows all point, eventually, toward the goal at $(3,3)$ — while routing *around* the pit at $(1,2)$ rather than past it.

**The policy is checkable, so check it rather than admiring it.** Look at column 3: states $(0,3)$, $(1,3)$, $(2,3)$ all say **↓**, walking straight down the right edge to the goal. Now look at $(0,2)$, directly above the pit: it says **→**, stepping sideways into the safe column instead of down. **The 10% slip probability is what makes that detour worth −0.02 per step** — a direct route past the pit risks a −1.0 with probability 0.1, and the arithmetic prefers the longer path. Remove the slip and the policy changes.

**Now the number that does not match the naive bound, and it is worth stopping on.** A $\gamma$-contraction should shrink the residual by $0.95$ per sweep, so reaching $10^{-12}$ would need about **540 sweeps**. It took **32**, implying an effective per-sweep factor near $0.42$. **The $\gamma$ bound is worst-case and this MDP beats it**, because the goal and pit are **absorbing**: once an episode terminates no value flows back, so the true contraction is $\gamma \times P(\text{not yet terminated})$. Episodes here end within a handful of steps, and the convergence rate reflects that.

**That is the general lesson about contraction bounds, not a quirk of this grid.** $\gamma$ guarantees convergence and bounds the *worst* case; the actual rate depends on the structure of the problem. **A method converging faster than its bound is normal; converging slower would be a bug.**

**Note where the discount factor is doing load-bearing work.** $\gamma = 0.95$ is what makes the Bellman operator a contraction and therefore makes the fixed point **unique and reachable**. It is not a statement about preferring near-term reward — it is what makes the problem well-posed. Set $\gamma = 1$ with no terminal state and values can diverge; the infinite sum need not converge at all.

**And note what value iteration required, because Session 3 is defined by not having it.** Every transition probability, every reward, for every state–action pair — `step_model` hands them over in closed form. **No agent in the world has that.** A robot does not know the probability that its wheel slips; a trading system does not know the market's transition kernel.

**Which is exactly why this $4\times4$ grid is the right size.** Because we own the model, we own $V^*$, $Q^*$, and $\pi^*$ **exactly** — a genuine answer key. Every learning algorithm in the remaining sessions is graded against it, including the value $V^*(\text{start}) = 0.642$ that the policy-gradient plot uses as its reference line. **Almost no RL tutorial can check its own answers**, and choosing a solvable environment over an impressive one is what makes that possible here.

---
### 🕐 Session 3 of 5 — *Temporal-Difference Learning* (~40 min)
**Goal:** learn the same values WITHOUT the model: TD(0) and Q-learning, checked against the oracle.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (policy gradients).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Temporal-Difference Learning</b></summary>

**Timing (~40 min).** 8 min what the agent no longer has · 12 min the TD error · 10 min the step-size condition · 10 min grading against the oracle.

**Open by removing the thing Session 2 relied on.** Value iteration needed the transition model — every probability, every reward, in advance. **An agent gets only experience: $(s, a, r, s')$ tuples, one at a time.** The question of this session is whether the Bellman equation is still usable when you cannot evaluate the expectation it contains.

**The answer is the session's one big idea, and it deserves to be stated as a reframing.** The Bellman equation was a *constraint*; violate it and the size of the violation is an **error signal**. $\delta = r + \gamma \max_a Q(s',a) - Q(s,a)$ is how surprised you are, and $Q \mathrel{+}= \alpha\delta$ moves toward consistency. **A single sample replaces the expectation, and averaging over many visits recovers it.**

**Then name the shape, because the room has now seen it three times.** $Q \mathrel{+}= \alpha \times \text{surprise}$ is the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup playing the role of the desired signal. Same in Session 1's incremental mean, same in stochastic gradient descent. **One update rule, four workshops** — and saying so explicitly is what turns a list of algorithms into a subject.

**Point at `alpha = 1.0 / N_sa[s,a]**0.6` and explain what the exponent buys.** Robbins–Monro requires $\sum\alpha = \infty$ (enough total movement to reach anywhere) and $\sum\alpha^2 < \infty$ (enough decay to stop bouncing). The exponent $0.6$ sits inside $(0.5, 1]$ and satisfies both. **Convergence here is a theorem with hypotheses, and the code satisfies them deliberately** — worth showing, because most tutorials use a constant $\alpha$ and then have no convergence guarantee at all.

**Explain off-policy learning in one sentence, since it is the reason this is Q-learning.** The update uses $\max_a Q(s',a)$ — the value of the **greedy** action — while the behaviour policy is ε-greedy and often acts randomly. **You learn about the optimal policy while following a different one**, which is what makes exploration and exploitation separable and is the property that makes replay buffers and offline RL possible.

**Set up the audit before the numbers appear.** Two things get measured: $\|Q - Q^*\|_\infty$, and whether the greedy action is optimal in every state. **Prepare the room for those two to disagree** — the value error is 0.232, which on a scale where values run 0 to 1 is not small, and the policy is nonetheless optimal in **100%** of states.

**That disagreement is the most useful idea in the session, so give it time.** The value function only has to get the **argmax** right, not the magnitudes. A state whose best action beats the runner-up by 0.3 tolerates 0.14 of error in each estimate without changing the decision. **Accurate values and correct policies are different objectives**, and control cares about the second. Ask where the residual 0.232 lives — the answer is rarely-visited state–action pairs, especially actions that walk into the pit, which the ε-greedy policy avoids and therefore samples poorly.

**Close on the honest boundary.** Tabular Q-learning has a convergence theorem because the table has one independent entry per state–action pair. **Replace the table with a neural network and the guarantee vanishes** — the "deadly triad" of function approximation, bootstrapping, and off-policy updates can diverge. DQN's replay buffers and target networks are engineering patches for the loss of exactly this theorem, and knowing that is what stops students expecting deep RL to behave like this cell.
</details>

## 4. Learning from the Surprise

💡 **Intuition.** Value iteration needed the transition model. An *agent* only gets experience: $(s, a, r, s')$. TD's move: use the Bellman equation as an **error signal** — the *TD error* $\delta = r + \gamma \max_a Q(s', a) - Q(s, a)$ is how surprised you are, and $Q \mathrel{+}= \alpha \delta$ is the [LMS update](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the 'desired signal'. Q-learning does this off-policy (learns the greedy value while exploring) — and, on a finite MDP with decaying exploration, provably converges to $Q^*$. We *check* that, since we own the oracle.

In [4]:
def env_step(s, a):
    outs = step_model(s, a)
    probs = [o[0] for o in outs]
    _, s1, r, d = outs[rng.choice(len(outs), p=probs)]
    return s1, r, d

Q = np.zeros((16, 4))
N_sa = np.zeros((16, 4))                                   # visit counts → decaying step size
eps = 1.0
errs = []
for ep in range(12000):
    s = rng.choice([s for s in range(16) if s not in (GOAL, PIT)])
    eps = max(0.05, eps * 0.9995)
    for _ in range(100):
        a = rng.integers(4) if rng.random() < eps else int(Q[s].argmax())
        s1, r, done = env_step(s, a)
        target = r + (0 if done else gamma * Q[s1].max())
        N_sa[s, a] += 1
        alpha = 1.0 / N_sa[s, a]**0.6                      # Robbins–Monro: Σα=∞, Σα²<∞ → convergence
        Q[s, a] += alpha * (target - Q[s, a])              # new = old + α·(surprise)
        s = s1
        if done: break
    if ep % 200 == 0: errs.append(np.abs(Q - Q_star).max())

plt.figure(figsize=(7.5, 2.6))
plt.semilogy(np.arange(len(errs))*200, errs)
plt.xlabel("episode"); plt.ylabel("‖Q − Q*‖∞")
plt.title("Q-learning converges to the DP oracle's Q* — from experience alone")
plt.tight_layout(); plt.show()
# tie-aware policy check: the learned greedy action must be (near-)optimal under Q*
nonterm = [s for s in range(16) if s not in (GOAL, PIT)]
optimal_choice = np.array([Q_star[s, Q[s].argmax()] >= Q_star[s].max() - 1e-3 for s in nonterm])
print(f"final ‖Q − Q*‖∞ = {np.abs(Q - Q_star).max():.3f}; learned greedy action optimal in {optimal_choice.mean():.0%} of states")

final ‖Q − Q*‖∞ = 0.232; learned greedy action optimal in 100% of states


/tmp/ipykernel_2966289/1394493565.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Q-learning, from experience alone with **no access to the transition model**, produced:

- $\|Q - Q^*\|_\infty = \mathbf{0.232}$
- learned greedy action optimal in **100%** of states

**Those two numbers disagree, and the disagreement is the most useful thing in the session.** On a scale where the true values run from 0 to 0.99, an error of 0.232 is **not small** — roughly a quarter of the range. Yet every single state's greedy action matches the oracle. **How can the values be that wrong and the policy be perfect?**

**Because a policy only needs the *argmax*, not the magnitudes.** If the best action at a state beats the runner-up by 0.3, then both estimates can be off by 0.14 without changing the decision. **Value accuracy and policy optimality are different objectives**, and control cares about the second. That is why the notebook's check is tie-aware — comparing $Q^*[s, \hat a]$ against $\max_a Q^*[s,a]$ rather than comparing $Q$ against $Q^*$ elementwise.

**Ask where the 0.232 actually lives, because the answer is diagnostic.** It sits in **rarely-visited state–action pairs** — above all, actions that step toward the pit. The ε-greedy policy avoids them once it has learned they are bad, so their counts stay low, their step sizes stay large, and their estimates stay noisy. **Q-learning converges where it looks**, and the residual is a map of where it stopped looking. This is Session 1's exploration/exploitation tension, still present with states attached.

**The convergence itself is not luck — the code satisfies a theorem's hypotheses on purpose.** `alpha = 1.0/N_sa[s,a]**0.6` gives $\sum\alpha = \infty$ (enough total movement to reach any value) and $\sum\alpha^2 < \infty$ (enough decay to stop bouncing) — the **Robbins–Monro** conditions. Most tutorials use a constant $\alpha$ and thereby forfeit any guarantee. **Convergence here is a theorem with hypotheses, and the exponent 0.6 is how they are met.**

**Note the update rule one more time, because it is now unmistakable.** `Q[s,a] += alpha * (target - Q[s,a])` is **new = old + α × surprise** — the LMS update from [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) with the Bellman backup as the desired signal, identical in shape to Session 1's incremental mean and to stochastic gradient descent. **Four workshops, one line.**

**And note the word "max" in the target, which is what makes this Q-learning rather than SARSA.** The update uses $\max_a Q(s',a)$ — the value of the **greedy** action — while behaviour is ε-greedy and frequently random. **You learn about the optimal policy while following a different one.** That off-policy property is what makes replay buffers, offline RL, and learning from logged human data possible at all.

**Finally, the boundary this result does not cross.** The convergence theorem holds because the table has one independent entry per state–action pair — 64 numbers, each updated by its own average. **Replace the table with a neural network and the guarantee disappears.** Function approximation, bootstrapping, and off-policy updates together form the "deadly triad", and their combination can diverge. Target networks and replay buffers in DQN are engineering patches for exactly the theorem that this tabular setting gets for free.

---
### 🕐 Session 4 of 5 — *Policy Gradients* (~40 min)
**Goal:** skip values, optimize the policy directly: REINFORCE with a baseline, from scratch.
**Builds on:** Session 3; [Training Dynamics](./Training_Dynamics.ipynb). &nbsp; **Feeds into:** Session 5 (the road to RLHF).

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Policy Gradients</b></summary>

**Timing (~40 min).** 10 min why skip values · 12 min the log-derivative trick · 10 min the baseline · 8 min reading the two curves.

**Motivate the change of target.** Sessions 2 and 3 learned *values* and derived a policy by taking argmax. But the thing you actually want is the **policy** — so why not optimise it directly? Two practical reasons the value route struggles: **continuous action spaces**, where the argmax is itself an optimisation problem, and **stochastic optimal policies**, which a greedy argmax cannot represent at all.

**Derive the log-derivative trick at the board, because it looks impossible before you see it.** You want $\nabla_\theta E_{\pi_\theta}[R]$, but the *distribution* depends on $\theta$ — the thing you are differentiating is the sampling, not the integrand. One line fixes it: $\nabla p = p\,\nabla\log p$, so $\nabla E[R] = E[R\,\nabla\log\pi(a|s)]$. **The gradient becomes an expectation you can sample.** Read the result in words: *reinforce the log-probability of what you did, in proportion to how well it went.*

**Then say plainly what the estimator costs.** It is **unbiased** and it is **extremely noisy** — the return $R$ multiplies the whole gradient, so a single lucky episode produces an enormous update. This is [SGD's](../Intro_Math/Optimization/Optimization.ipynb) noise-floor problem with the noise amplified by the reward scale.

**The baseline is the fix and it deserves a real derivation, because "subtract the mean" sounds arbitrary.** $E[b\,\nabla\log\pi] = b\,\nabla\!\!\int\!\pi = b\,\nabla 1 = 0$. **Any baseline not depending on the action leaves the gradient unbiased** — you get variance reduction for free, with no bias to pay for it. That is rare enough to be worth emphasising: most variance reductions cost bias, and this one does not.

**Give the intuition alongside the algebra.** Without a baseline, every action taken in a good episode is reinforced — including the bad ones that happened to co-occur with success. With a baseline, only **better than average** actions are reinforced and worse-than-average ones are actively pushed down. **The signal changes from "was this good?" to "was this better than usual?"**, which is a far more informative question.

**Set expectations for the plot honestly.** Both curves reach the oracle line at $V^*(\text{start}) = 0.64$. **The baseline does not get to a better place — it gets there more calmly.** Point at the *width* of the two traces rather than their endpoints; the variance reduction is visible as smoothness, and reading the wrong feature of the plot is the easy mistake here.

**Note the two limitations of this implementation, since students will otherwise generalise from a favourable case.** It is **on-policy** — every episode is used once and discarded, which is why policy gradients are sample-hungry compared to Q-learning's replay. And it is **tabular**, with 64 independent parameters; the same algorithm on a neural policy inherits every difficulty from [Training Dynamics](./Training_Dynamics.ipynb) plus a moving data distribution.

**Close by naming what PPO adds, so Session 5 is a short step.** REINFORCE + a learned baseline (a critic) + a **trust region** that refuses updates moving the policy too far in one step. **All three are variance and stability armour bolted onto the estimator derived here**, and none of them changes the underlying idea.
</details>

## 5. Differentiating Through Luck

💡 **Intuition.** Values are a detour; why not adjust the policy's parameters to make good episodes more likely? The log-derivative trick makes the un-differentiable differentiable: $\nabla E[R] = E[R \, \nabla \log \pi(a|s)]$ — *reinforce the log-probability of what you did, in proportion to how well it went*. The estimator is unbiased but wildly noisy ([SGD's](../Intro_Math/Optimization/Optimization.ipynb) noise-floor problem, squared); subtracting a **baseline** (the mean return) cancels variance without adding bias — the single most important practical trick in policy-land.

In [5]:
# REINFORCE on the same gridworld (tabular softmax policy)
def run_episode(theta, max_steps=60):
    s = 0; traj = []
    for _ in range(max_steps):
        logits = theta[s]
        p = np.exp(logits - logits.max()); p /= p.sum()
        a = rng.choice(4, p=p)
        s1, r, done = env_step(s, a)
        traj.append((s, a, r))
        s = s1
        if done: break
    return traj

def reinforce(use_baseline, iters=1500, lr=0.15):
    theta = np.zeros((16, 4)); returns_hist = []
    for it in range(iters):
        traj = run_episode(theta)
        G = 0.0; Gs = []
        for (_, _, r) in reversed(traj):
            G = r + gamma * G; Gs.append(G)
        Gs = Gs[::-1]
        b = np.mean(Gs) if use_baseline else 0.0
        for (s, a, _), G_t in zip(traj, Gs):
            p = np.exp(theta[s] - theta[s].max()); p /= p.sum()
            grad = -p; grad[a] += 1                       # ∇ log softmax
            theta[s] += lr * (G_t - b) * grad
        returns_hist.append(Gs[0])
    return np.array(returns_hist)

plt.figure(figsize=(8, 2.8))
for name, ub in [("REINFORCE", False), ("REINFORCE + baseline", True)]:
    h = reinforce(ub)
    sm = np.convolve(h, np.ones(50)/50, "valid")
    plt.plot(sm, label=f"{name} (final ≈ {sm[-1]:.2f})")
plt.axhline(V[0], color="k", linestyle=":", linewidth=0.9, label=f"oracle V*(start) = {V[0]:.2f}")
plt.legend(fontsize=8); plt.xlabel("episode"); plt.ylabel("return from start")
plt.title("policy gradient reaches the DP oracle's value; the baseline gets there calmer")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2966289/2260655531.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two REINFORCE runs, both climbing to the dotted line at $V^*(\text{start}) = \mathbf{0.64}$ — the value that Session 2's dynamic programming computed **exactly**, without any learning. A policy-gradient method that never estimated a value function, never used the transition model, and only ever sampled episodes, arrived at the optimum.

**Read the right feature of the plot: the difference is *width*, not height.** Both curves reach roughly the same place. **The baseline does not find a better policy — it finds the same one more calmly.** Compare the raggedness of the two traces; that is the variance reduction, and it is the whole content of the comparison. Reading the endpoints instead of the smoothness is the easy mistake.

**Why the baseline works is one line of algebra, and it is worth doing because "subtract the mean" sounds arbitrary.**

$$E[b\,\nabla\log\pi] = b\,\nabla\!\!\int\!\pi\,da = b\,\nabla 1 = 0$$

**Any baseline that does not depend on the action leaves the gradient exactly unbiased.** So variance falls and nothing is paid for it. That is unusual — most variance reductions cost bias — and it is why the baseline is the single most important practical trick in policy-gradient methods.

**The intuition is sharper than the algebra.** Without a baseline, *every* action taken during a successful episode gets reinforced, including the bad ones that merely co-occurred with success. With a baseline, only **better-than-average** actions are pushed up and worse-than-average ones are pushed down. **The learning signal changes from "was this good?" to "was this better than usual?"**

**Note what made the whole method possible: the log-derivative trick.** The objective is an expectation whose *distribution* depends on $\theta$, so you cannot differentiate the integrand and be done. But $\nabla p = p\nabla\log p$ turns it into $\nabla E[R] = E[R\,\nabla\log\pi(a|s)]$ — **an expectation you can sample**. The line `grad = -p; grad[a] += 1` is exactly $\nabla\log\text{softmax}$, and everything else in `reinforce` is bookkeeping around it.

**Now the honest caveats, because this demo is a favourable case.** It is **tabular**: 64 independent parameters, no function approximation, no generalisation required. It is **on-policy**: every episode is used once and thrown away, which is why policy gradients are far more sample-hungry than Q-learning with replay. And the run is a **single seed** — REINFORCE is high-variance enough that another seed can look noticeably different, so the smoothness comparison would be more convincing averaged over five.

**The comparison against the oracle is what makes this cell evidence rather than a demonstration.** Session 2's exact value $V^*(\text{start}) = 0.642$ is the answer key; without it, a converged-looking curve tells you the algorithm stopped moving, not that it stopped in the right place. **Almost no RL experiment can perform this check** — and the habit of asking "converged to *what*?" is what this workshop is really teaching.

**Finally, note how little separates this from what trains modern language models.** PPO is REINFORCE plus a learned baseline (a critic) plus a trust region that refuses over-large updates. **Both additions are variance and stability armour on the estimator you just watched work**, and neither changes the underlying idea. Session 5 makes that translation explicit.

---
### 🕐 Session 5 of 5 — *The Road to RLHF* (~30 min)
**Goal:** connect this course to modern practice: reward models, KL anchors, and PPO's role.
**Builds on:** Session 4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 5: The Road to RLHF</b></summary>

**Timing (~30 min).** 8 min the translation table · 8 min the reward model · 7 min the KL anchor · 7 min reward hacking and DPO.

**This session has no code, and its job is translation.** The room now owns bandits, MDPs, TD learning, and policy gradients on a $4\times4$ grid. **Everything in RLHF is those four ideas at a scale where none of them can be verified**, and the value of this session is that students can now name each component rather than treating RLHF as a black box.

**Do the mapping slowly and make the room supply the terms.** Policy = the language model. State = prompt plus text generated so far. Action = the next token. Episode = a completion. **Then let the strangeness land: the state space is every possible text prefix, and the action space is the vocabulary.** A $4\times4$ grid has 16 states; this has more than atoms in the universe. Nothing about the algorithms changes; everything about the verification does.

**Spend real time on where the reward comes from, because it is the part with no analogue in the earlier sessions.** The gridworld handed out $+1$ and $-1$ by fiat. **There is no reward function for "a good answer."** So one is *learned*: humans compare pairs of completions, and a reward model is trained by cross-entropy on which was preferred — straight [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb). **The reward is a fitted model, with all the fallibility that implies**, and that single fact generates the rest of the session.

**Which makes reward hacking predictable rather than surprising, and the room already has the vocabulary.** The policy optimises the reward model. The reward model was fitted on human preferences over *the distribution of completions that existed when it was trained*. Optimising hard pushes the policy **off that distribution** — into exactly the region where [Uncertainty in ML](./Uncertainty_in_ML.ipynb) showed models are confidently wrong. **Reward hacking is optimisation exploiting an out-of-distribution model**, and the OOD workshop measured how badly that fails: 9% of the off-map region flagged, 90% confidently mis-scored.

**The KL anchor is then the obvious defence, and it is worth deriving as one.** Penalise the policy for drifting from the pretrained model. Read it as a **trust region on the whole model**: improve preferences *without leaving the region where the reward model was fitted*. Same logic as PPO's clipped update, one level up — and both are answers to "how far can I trust this estimate?"

**Frame PPO as a three-part assembly, since the room built two of the parts.** REINFORCE (Session 4) + a learned baseline, i.e. a critic (Session 4's variance argument) + a trust region (new). **Nothing conceptual is added; it is variance and stability armour**, which is exactly what a $10^{11}$-parameter policy with an expensive, imperfect reward signal needs.

**Explain why DPO displaced the pipeline, because it is a good closing insight.** If the reward model is fitted from preference pairs and then optimised under a KL constraint, the two steps can be algebraically composed into **one supervised loss on the preference pairs directly** — no reward model, no rollouts, no RL loop. **Most of RLHF's engineering difficulty was the RL, and DPO removes it.** The preferences still matter; the machinery does not.

**Close on the epistemic point that ties the workshop together.** Every algorithm in Sessions 1–4 was checked against an exact oracle. **At RLHF scale there is no oracle** — no $V^*$, no $Q^*$, no ground-truth reward — and evaluation falls back on held-out preference accuracy and human judgment, both noisy and both gameable. **The habit of asking "converged to what?" is worth more the further you get from a $4\times4$ grid**, precisely because nothing will answer it for you.
</details>

## 6. From Gridworld to Chatbots

The [LLM workshop](./LLMs_from_the_Ground_Up.ipynb) said 'preference tuning shapes judgment'; you now have the vocabulary for how:

1. **The policy** is the language model; a *state* is the prompt + text so far, an *action* is the next token, an *episode* is a completion.
2. **The reward** comes from a *reward model* trained on human preference pairs — [cross-entropy](../Intro_Math/Information_Theory/Information_Theory.ipynb) on 'which answer did the human prefer'.
3. **The optimizer** is a policy gradient with variance-reduction armor: PPO ≈ REINFORCE + a learned baseline (critic) + a *trust region* (clipped updates — don't move the policy further than the reward model's validity extends).
4. **The KL anchor**: reward is penalized by KL divergence from the pretrained model — 'improve preferences *without leaving the language manifold*'. DPO folds reward model + RL into one supervised loss on preference pairs, which is why it took over.

💡 **Intuition.** Everything hard about RLHF is Session 4's variance problem wearing a $10^{11}$-parameter costume, plus one new failure mode this course equips you to name: **reward hacking** — the policy exploiting the reward model where it's [off-distribution](./Uncertainty_in_ML.ipynb).

## 7. Conclusion

Bandits isolate exploration; Bellman makes value self-consistent; TD learns it from surprise (LMS's heartbeat again); policy gradients differentiate through luck with a baseline as armor; RLHF is all four at industrial scale. Every algorithm here was checked against an exact oracle — a habit worth keeping when the environments stop being 4×4.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — the policy being tuned.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — the SGD theory under the noise.
- [Beyond Kalman](../Intro_Time_Series/Beyond_Kalman.ipynb) — belief-state tracking: what 'state' means when you can't see it.